<a href="https://www.kaggle.com/code/soumyajitworkspace/gpt-ac?scriptVersionId=308003934" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F 
import math






def scaled_dot_product_attention(Q,K,V,mask = None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q,K.transpose(-2,-1))
    scores = scores / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    weigths = F.softmax(scores,dim=-1)
    return torch.matmul(weigths,V), weigths






class CausalMultiheadattention(nn.Module):
    def __init__(self,d_model,num_heads,dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model,d_model)
        self.W_V = nn.Linear(d_model,d_model)
        self.W_O = nn.Linear(d_model,d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self,x,batch):
        x = x.view(batch,-1,self.num_heads,self.d_k)
        return x.transpose(1,2)
    def forward(self,x,mask=None):
        batch = x.size(0)
        Q = self.split_heads(self.W_Q(x), batch)
        K = self.split_heads(self.W_K(x), batch)
        V = self.split_heads(self.W_V(x), batch)
        out, _ = scaled_dot_product_attention(Q,K,V,mask)
        out = out.transpose(1,2).contiguous()
        out = out.view(batch,-1,self.d_model)
        return self.W_O(out)
    






class FeedForward(nn.Module):
    def __init__(self,d_model,d_ff,dropout= 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
            

        )

    def forward(self,x):
        return self.net(x)
    







class gptblock(nn.Module):
    def __init__(self,d_model,num_heads,d_ff,dropout=0.1):
        super().__init__()
        self.attention = CausalMultiheadattention(d_model,num_heads,dropout)
        self.ff = FeedForward(d_model,d_ff,dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self,x,mask=None):
        x = x + self.attention(self.norm1(x),mask)
        x = x + self.ff(self.norm2(x))
        return x

        







class minigpt(nn.Module):
    def __init__(self,vocab_size ,
                 d_model = 256,
                  num_heads = 8,
                   num_layers = 6,
                    d_ff = 1024,
                     max_len = 1024,
                      dropout = 0.1 ):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size,d_model)
        self.pos_embedding = nn.Embedding(max_len,d_model)
        self.dropout = nn.Dropout(dropout)
        self.block = nn.ModuleList([
            gptblock(d_model,num_heads,d_ff,dropout)
            for _ in range(num_layers)

        ])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model,vocab_size,bias= False)
        self.lm_head.weight = self.token_embedding.weight
        self.apply(self._init_weights)
        
    def _init_weights(self,module):
        if isinstance(module,nn.Linear):
            nn.init.normal_(module.weight, mean = 0.0,std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module,nn.Embedding):
            nn.init.normal_(module.weight,mean=0.0,std=0.2)
    def forward(self,idx,targets=None):
        batch, seq_len = idx.shape
        device = idx.device
        token_emb = self.token_embedding(idx)
        pos = torch.arange(seq_len,device=device)
        pos_Emb = self.pos_embedding(pos)
        x = self.dropout(token_emb + pos_Emb)
        mask = torch.tril(
            torch.ones(seq_len,seq_len,device=device)

        )

        for block in self.block:
            x = block(x,mask)
        x = self.norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1,logits.size(-1)),
                targets.view(-1)

            )
        return logits,loss

    @torch.no_grad()
    def generate(self,idx,max_new_tokens,temperature=1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -1024:]
            logits, _ = self(idx_cond)
            logits  = logits[:,-1,:]
            logits = logits / temperature
            probs = F.softmax(logits,dim=-1)
            idx_next = torch.multinomial(probs,num_samples=1)
            idx = torch.cat([idx,idx_next], dim=1)
        return idx


















device = "cuda" if torch.cuda.is_available() else "cpu"
vocab_size = 50214

model = minigpt(
    vocab_size= vocab_size,
    d_model = 256,
    num_heads=8,
    num_layers=6,
    d_ff= 1024,
    max_len = 1024
).to(device)

params = sum(p.numel() for p in model.parameters())
print(f"MiniGPT Parameters: {params:,}")
print(f"~{params/1e6:.1f}M parameters")

x = torch.randint(0,vocab_size,(2,32)).to(device)
targets = torch.randint(0,vocab_size,(2,32)).to(device)

logits, loss = model(x, targets)
print(f"Input:   {x.shape}")
print(f"Logits:  {logits.shape}")
print(f"Loss:    {loss.item():.4f}")

# Test generation!
print("\nTesting generation...")
start  = torch.zeros((1, 1), dtype=torch.long).to(device)
output = model.generate(start, max_new_tokens=20, temperature=0.8)
print(f"Generated shape: {output.shape}")
print("MiniGPT works! ✅")






MiniGPT Parameters: 17,856,000
~17.9M parameters
Input:   torch.Size([2, 32])
Logits:  torch.Size([2, 32, 50214])
Loss:    10.9510

Testing generation...
Generated shape: torch.Size([1, 21])
MiniGPT works! ✅
